In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 7.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
google_gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://baritone-scarily-unmade.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://baritone-scarily-unmade.ngrok-free.dev


True

In [6]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    print("EVENT: ", event)
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[TextMessage(text=event.message.text),#第一次
                            TextMessage(text=event.message.text)]#第二次
            )
        )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"U119d07e95858d5d83008458c2d433c76","events":[{"type":"message","message":{"type":"text","id":"616072775626654408","quoteToken":"oRNke5SZEqecKsD8Xv2RKn3GHN10alWFoIRgDdzmz39sK9xM-v4-SBsr2t9aY8wQ4WBgZve7zmyMzbQ8tRp6GKn6yaH_EXUeijvdFUJJ98gI_qUF_is6md41wXVmLIWIUPJFVt9A9tc7-lwDCMpyhA","markAsReadToken":"oXop9toRylmM9-KOAFYCGfoq9ZZdzKc3Frxpda1quOU4l5mDDFUlGdjCFMpGZgckyRle2RuKProvNRKyLlBfsz4Wjyq6PdTBAdWdKSageOXEN70qEhJuvl_fr1Ju8oioFThyK6t2NSDKyTKh7vJdlYbfe_cBNqDzK9EVrhRhfoEh1WE1XUnr6aYfRXdsoJnCvGGnDZdE8l1H6rMcOrKIYw","text":"z"},"webhookEventId":"01KSS9R72JYMEKP5DG28VFNBNG","deliveryContext":{"isRedelivery":false},"timestamp":1780039293664,"source":{"type":"user","userId":"U3d0a7956380119affa33de4fb575bbcb"},"replyToken":"562b7d3dbb474dfaa81d87e7589ea0c4","mode":"active"}]}
EVENT:  type='message' source=UserSource(type='user', user_id='U3d0a7956380119affa33de4fb575bbcb') timestamp=1780039293664 mode=<EventMode.ACTIVE: 'active'> webhook_event_id='01KSS9R72JYMEKP5DG28VFNBN

INFO:werkzeug:127.0.0.1 - - [29/May/2026 07:21:34] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U119d07e95858d5d83008458c2d433c76","events":[{"type":"message","message":{"type":"text","id":"616072794249363815","quoteToken":"jvj776xvwhwyof5LN15VSF2JYvGjWPG-JF0UbGEjDuNLu2YRe25p8KPmn_4D4-eFUKT_XEqNunU86aLJ7zkO1n-GYUHAh_aodYEnizZDrK-9a6mHxOvJecD_GxXKg2XIi1cO6wQ0klaqxW-R70m7BA","markAsReadToken":"7TqbDo6BSheGjYfe9q_7rDDwj9iYLJgtCrgdEhA16A3Aa01dyMWgCD0N4DF6oQKzQ5ZACiaxvX_lbaohmvEWJmq0Qx0frCwUCQe18fkYrIRyjqh4JGoBfbcHvS7Nz332jmbteh6OKWXiMfOzcyX44xVrSc_G3ODdoD0-TEvu1emFiobIqVBFWw7MwD7nImFsNATVsdX4JprjY0qjobFocQ","text":"Nigachu"},"webhookEventId":"01KSS9RHTGTCXRAZF20ENJF7B2","deliveryContext":{"isRedelivery":false},"timestamp":1780039304689,"source":{"type":"user","userId":"U3d0a7956380119affa33de4fb575bbcb"},"replyToken":"a9c70d4fcd804cf487023cf27032bd18","mode":"active"}]}
EVENT:  type='message' source=UserSource(type='user', user_id='U3d0a7956380119affa33de4fb575bbcb') timestamp=1780039304689 mode=<EventMode.ACTIVE: 'active'> webhook_event_id='01KSS9RHTGTCXRAZF20

INFO:werkzeug:127.0.0.1 - - [29/May/2026 07:21:46] "POST / HTTP/1.1" 200 -
